# Theory builder

A point-and-click form for declaring stochastic field theories.  Fill in the tabs, hit **Save theory file**, and a `.theory.py` lands in `theories/` ready to be loaded by `theory_runner.ipynb`.

## What this is for

This UI builds field theories in the **MSR-JD formalism** (Martin&ndash;Siggia&ndash;Rose / Janssen&ndash;de Dominicis): given the action of a stochastic ODE/SDE, the underlying framework can compute its mean-field, multi-point cumulants, loop corrections, and cross-correlations.

It handles theories of the form $\dot x = -\partial H/\partial x + \eta$ and their generalizations &mdash; nonlinear dynamics, multi-population systems, colored noise, cross-correlated noise.

## Workflow

1. **Open the form** (cell below).
2. **Fill the tabs left to right.**  Most theories use only a subset:
   - Always needed: **Model**, **Fields**, **Parameters**, **Action**, **MF**, **Defaults**
   - Skippable: **Populations** (scalar theories don't need it), **Functions** (only for non-polynomial nonlinearities like `tanh`), **Kernels** (only for memory / convolution terms), **Noise** (white Gaussian can go straight in the Action)
3. **Watch the sidebar.**  The panel just under the header surfaces undeclared-name errors in your action, mismatched populations, syntax errors with line numbers, etc.  It stays quiet on a clean spec.
4. **Pre-compute** (recommended).  The button next to Save validates the action, builds the propagator, solves the saddle, and warms the cache.  Catches problems before a long run.
5. **Save theory file.**  Writes `theories/<your-theory-name>.theory.py`.
6. **Switch to the runner.**  In `theory_runner.ipynb`, set `THEORY_NAME = '<your-theory-name>'` (filename without `.theory.py`) and execute.

---

## Per-tab guide

### 1. Model

Just a name + optional description.  The name is slugified into the `.theory.py` filename on save (e.g. `"Damped Pendulum"` &rarr; `damped_pendulum.theory.py`).

### 2. Populations  *(skip for scalar theories)*

A *population* is a group of multiple identical units that share the same dynamics &mdash; e.g. *N* neurons in a network, *N* spins on a lattice.  Declare each group's name + integer size.

**Skip this tab** if every variable in your theory is a single scalar.

**Example**: a 2-population neural network would declare `E (size 4)` and `I (size 3)`.

### 3. Fields

The physical variables of your theory &mdash; what you'd write on the left of an SDE.  For each declared field `x` the framework automatically creates two companions you'll use elsewhere:

| Symbol | What it is | Where it appears |
|---|---|---|
| `x` | the field itself | the action's deterministic part, MF equations |
| `xt` | MSR *response* field (auxiliary) | the action's response-bilinear and noise pieces |
| `xstar` | the *saddle* (steady-state value) | the MF equations solve for this |

Leave **population** blank for a scalar field.  Attach a population to make `x`, `xt`, `xstar` all vectors of that population's size &mdash; then you use `x[i]` indexing throughout.

**Example (scalar)**: declare `x`, population blank.  Use `x`, `xt`, `xstar` in the action / MF.

**Example (indexed)**: declare `x` with population `A`.  Use `x[i]`, `xt[i]`, `xstar[i]`, wrap in `sum(... for i in A)`.

### 4. Parameters

Any numerical knob your action / MF refer to &mdash; drift coefficients, coupling strengths, noise amplitudes, time constants.

**Shape** (controlled by the two index dropdowns):

| `index_1` | `index_2` | Shape | How you write it |
|---|---|---|---|
| blank | blank | scalar | `mu` |
| `A`   | blank | vector of size *N_A* | `mu[i]` |
| `A`   | `B`   | matrix *N_A &times; N_B* | `w[i, j]` |

**`domain`** is a hint to the Newton solver: `positive` &rarr; search `[0, 5&middot;scale]`, `real` &rarr; `[-3&middot;scale, 3&middot;scale]`, blank &rarr; no preference.

**`default`** is the value the runner uses if you don't override it.  Match the shape:
- scalar: `1.0`
- vector: `[1.0, 2.0]`
- matrix: `[[1.0, 0.5], [0.5, 1.0]]`

### 5. Functions  *(skip unless needed)*

Use this tab for **non-polynomial** transformations of fields &mdash; e.g. `tanh(x)`, `exp(x)`, `sigmoid(x)`, custom transfer functions.  Polynomials like `x^3` go directly in the action.

Each row gives the function a name and an expression body.  Then call it in the action like a normal Python function: `f(x)` for a global function, `f[i](x[i])` if it's indexed by a population.

**Example**: a `tanh`-shaped transfer

| field | value |
|---|---|
| name | `f` |
| population | (blank) |
| args | `x` |
| expression | `tanh(a*x)` |

Then in the action: `xt * (Dt*x + mu*x - f(x))`.

The framework Taylor-expands the function around the saddle value `xstar` automatically &mdash; you don't write the expansion yourself.

### 6. Kernels  *(skip unless needed)*

Memory kernels for **convolution terms** in the action &mdash; i.e. one field filtered through a kernel before coupling to another.

Provide either:
- `time_expr` &mdash; the kernel as a function of `t` (and parameters); use `heaviside(t)` for causality
- `freq_image` &mdash; the Fourier image (function of `omega`); **preferred**, used directly by the propagator builder

**Example**: a single-exponential memory kernel $K(t) = (1/\tau_k)\,e^{-t/\tau_k}\,\Theta(t)$

| field | value |
|---|---|
| name | `K` |
| index_1 / index_2 | (blank) |
| time_expr | `(1/tauk)*exp(-t/tauk)*heaviside(t)` |
| freq_image | `1/(1 + I*omega*tauk)` |

Then in the action: `xt * Conv(K, y)` couples *x* to the kernel-filtered version of *y*.

### 7. Noise

The stochastic forcing.  Each row declares one piece of the noise correlator
$$\langle \eta(t)\,\eta(t') \rangle = \text{coefficient} \times \text{kernel}(t - t').$$

**Skip this tab** if your action already includes the noise term as an explicit `D*xt^2`-style line (fine for white Gaussian).  **Use it** for colored noise, cross-correlated noise, or higher-order shot-noise cumulants.

Three concrete examples for $dx/dt = -\mu x + \eta$:

| Noise | row spec |
|---|---|
| White Gaussian, $\langle \eta\eta\rangle = 2D\,\delta(\tau)$ | `name=Cxx, legs=xt, order=2, coeff=2*D, kernel=dirac_delta(tau)` |
| OU-colored, $\langle \eta\eta\rangle = (D/\tau_c)\,e^{-\lvert\tau\rvert/\tau_c}$ | `name=Cxx, legs=xt, order=2, coeff=D/tauc, kernel=exp(-abs(tau)/tauc)` |
| 2D cross-correlated, $\langle \eta_x\eta_y \rangle = \rho\sqrt{D_1 D_2}\,\delta(\tau)$ | `name=Cxy, legs=xt, yt, order=2, coeff=rho*sqrt(D1*D2), kernel=dirac_delta(tau)` |

### 8. Action

The MSR-JD action $S$ as a single expression.  Shape: **response field $\times$ (equation of motion)**, possibly plus a noise piece.

**1D Langevin example** ($dx/dt = -\mu x - \varepsilon x^3 + \eta$, $\langle \eta\eta\rangle = 2D\,\delta$):

```
xt * ((Dt + mu) * x + eps * x^3) - D * xt^2
```

**Indexed (population) example**: wrap each term in a Python comprehension over the population.

```python
sum(xt[i] * ((Dt + mu[i]) * x[i] + eps * x[i]^3) - D[i] * xt[i]^2
    for i in A)
```

**Matrix coupling** between fields via a matrix parameter `w` indexed by `A, A`:

```python
sum(xt[i] * (-sum(w[i,j]*x[j] for j in A)) for i in A)
```

**Conventions**:
- Write physical fields directly (`x`, `v`, &hellip;).  The framework handles the saddle + fluctuation expansion under the hood &mdash; don't write `xstar + dx` yourself.
- `Dt` is the time derivative: `(Dt + mu)*x`, `(tau*Dt + 1)*v`.
- Custom functions from tab 5 work like Python calls: `f(x)`, `f[i](x[i])`.
- Matrix entries: `w[i, j]` or `w[i][j]` &mdash; both parse equivalently.
- Noise can go here (`-D*xt^2`) or on the Noise tab &mdash; pick whichever is cleaner.

### 9. MF (mean-field equations)

The equations whose saddle solutions give the steady-state values `xstar`.  One row per equation, `LHS = RHS`.

`Dt` is allowed on either side.  At the saddle the solver sets `Dt &rarr; 0`, then runs multi-start Newton over every distinct root.

**Example**: $dx/dt = -\mu x - \varepsilon x^3 \Rightarrow$ row `lhs=(Dt + mu)*x, rhs=-eps*x^3`.

**Picking a root** when there are several (bistable, etc.): the solver sorts them in ascending order and `fixed_point_index` (set on this tab) picks which to expand around.  The default is `0` (smallest).

**Stability filter** (checkbox below the table):
- **ON**: keep only the linearly stable roots before indexing &mdash; what you want for a bistable theory where the unstable branch is irrelevant.
- **OFF** (default): use every root.  Leave off when none of your equations contain `Dt` (purely algebraic system) &mdash; there's nothing to score.

### 10. Defaults

Suggestions the runner picks up by default.  Each is overridable per-run; nothing here changes the physics.

| Knob | Meaning |
|---|---|
| `k` | number of external legs in the cumulant.  `1` = mean, `2` = covariance / power spectrum, `3` = third cumulant, &hellip; |
| `max_ell` | loop order.  `0` = tree (LNA), `1` = +1-loop, `2` = +2-loop.  Cost rises rapidly. |
| `tau_max`, `tau_step` | time-lag grid for two-point quantities $C(\tau)$ |
| External legs | which observables to correlate.  Two legs on the same field = auto-correlation; legs on different fields = cross-correlation. |

---

## Restrictions and gotchas

- **No `Conv(...)` in MF equations.**  The stationary mean-field assumption already collapses convolutions of constants; put memory kernels only in the Action.
- **`max_ell >= 1` is slow with colored noise.**  Each ell=1 diagram does an extra $\tau$-integration.  With 150+ diagrams and a full $\tau$-grid, expect minutes to hours.  Set `USE_GROUPED_PHASE_J=True` in the runner notebook to accelerate substantially.
- **Bistable theories**: turn the stability filter ON (MF tab) so the diagrammatic expansion sits at a *stable* saddle.  Around an unstable saddle the series isn't defined.
- **Large $|\mu|$ in non-symmetric regimes**: Phase J runtime grows fast.  Stick to $|\mu| \lesssim 1.5$ unless you've benchmarked the specific theory.
- **Noise tab declares only the cumulants you list.**  If you write only an order-2 row, the noise is purely Gaussian.  Non-Gaussian noise requires explicit higher-order rows.
- **Saved files are Python source.**  Hand-editing a `.theory.py` is fine; the load-edit-save round-trip preserves everything the UI exposes (plus any custom LaTeX strings carried in the file).

## Tips

- The **sidebar** at the top of the form is your live error-checker.  It reports undeclared names in the action, comprehensions iterating over non-existent populations, syntax errors with line numbers, and parameters/fields referencing missing populations.  If it's silent, you're in good shape.
- The **Pre-compute** button runs the full sanity stack: action parse, propagator build, saddle solve, cache warm.  Cheap (~1&ndash;3 s for most theories) and catches problems before a long run.
- The **Load theory file** dropdown lets you re-open any previously-saved `.theory.py` for editing &mdash; round-trip preserves everything.
- **Reset** is destructive (wipes the form).  Save first if you might want to come back.

---

## Launch the form

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from api.ui import TheoryUI

ui = TheoryUI()
ui.show()

---

## After saving

The form's spec stays accessible on the `ui` object:

```python
ui.spec()        # the form state as a dict
ui.last_saved    # path of the most recent save
```

To compute with your freshly-saved theory, open `theory_runner.ipynb` and set

```python
THEORY_NAME = '<your-theory-name>'   # the filename minus '.theory.py'
```

in the runner's configuration cell.